# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdsvr/flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 4 — CTR / Engagement Opportunity Scoring** (provisional; I'll confirm or change by end of Week 4).

I picked it because my Week-1 discovery in notebook 01 points straight at it: when I held position tier constant, **comparison articles captured far fewer clicks than keyword articles at the same position** (~0.14% vs ~0.35% mean CTR on page 1 — the code below recomputes it). That means "low CTR" is not one number — it depends on position, content type, and volume at the same time, which is exactly the kind of tangled pattern Lane 4 is about. My fallback is Lane 2 (Refresh Scoring), since I've already run that pipeline end to end, but Lane 4 is the question my own evidence raised.

In [1]:
# The number behind the lane choice: CTR by content type at the SAME position tier.
# Data gotchas applied (docs/data-dictionary.md): rate columns are x100 percentages
# (ctr = 0.76 means 0.76%), and avg_position == 0 means "no data", not rank zero.
import os
import pandas as pd
import numpy as np

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("../..")  # work/notebooks/ -> repo root

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

visible = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)]
page1 = visible[visible["position_tier"] == "page_1"]
by_type = page1.groupby("content_type")["ctr"].agg(mean_ctr_pct="mean", n="count").round(3)
print("Mean CTR (%) on page-1 positions, impressions >= 100:")
print(by_type.to_string())

Mean CTR (%) on page-1 positions, impressions >= 100:
                    mean_ctr_pct     n
content_type                          
comparison article         0.141   226
feedly article             0.905   221
keyword article            0.346  8186


## 2. The question: decision, action, cost of a wrong call

**The question:** *Which visible pages under-capture clicks for the position they already hold, and which of them should an editor review first this cycle?*

**The frame (one paragraph):** For a content editor with a fixed review budget, deciding *which under-clicking pages to review first*, I will build a **ranked opportunity queue** from observable search signals (impressions, position, CTR, content type, intent, age), scoring each page's **CTR gap versus what's expected for its position tier and volume**, measured by **precision@50** (the metric is named now, before any model exists). A wrong call costs a wasted review slot — and, worse, the real underperformer that slot should have gone to. A plain rule isn't enough because the rule already exists and doesn't decide anything: the starter pipeline's own threshold (impressions >= 500, position 1-20, CTR < 0.5%) flags **9,759 pages** — a to-do list 200x larger than anyone's capacity. Ranking *within* that pool is the actual problem. I will claim only observed / directional / decision-support results.

- **Unit of analysis:** one pseudonymized content item per client (one CSV row; later, per-content aggregates from the warehouse daily facts).
- **Output:** a ranked list with a score and human-readable reason codes (e.g. *high impressions, page-1 position, CTR far below tier norm*).
- **Who acts, and how:** an editor takes the top ~50 and, per page, rewrites the title/meta, fixes the intent match, improves snippet structure — or explicitly marks it "monitor".
- **Cost of a wrong recommendation:** one wasted editor review (~top-50 slot), plus the missed opportunity of the page that should have ranked there. False positives waste hours; false negatives quietly leave ~91M 90-day impressions under-clicked.
- **Why data/ML helps:** "low CTR" is only meaningful relative to position, volume, content type, and intent *simultaneously* — the thresholds are tangled and shift by client. That's a pattern too messy to hand-write, which is exactly when a learned ranking earns its place (and it must prove it beats the transparent rule, or the rule wins).

In [2]:
# Why this is a ranking problem, not a rule: the rule's to-do list vs anyone's capacity.
# Threshold = the starter pipeline's own ctr_review_candidate definition (scripts/04).
pool = df[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0) & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)  # 0.5% — remember, ctr is a x100 percentage
]
REVIEW_CAPACITY = 50  # pages one editor can realistically inspect per cycle

print(f"Pages the plain rule flags:        {len(pool):,}")
print(f"Editor capacity per cycle:         {REVIEW_CAPACITY}")
print(f"Rule output vs capacity:           {len(pool) / REVIEW_CAPACITY:.0f}x too many")
print(f"90d impressions sitting in pool:   {int(pool['impressions_90d'].sum()):,}")
print()
print("The rule answers 'which pages qualify?' — it cannot answer 'which 50 first?'.")
print("That ordering decision is the project, scored by precision@50.")

Pages the plain rule flags:        9,759
Editor capacity per cycle:         50
Rule output vs capacity:           195x too many
90d impressions sitting in pool:   90,968,008

The rule answers 'which pages qualify?' — it cannot answer 'which 50 first?'.
That ordering decision is the project, scored by precision@50.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter CSV, all computed live below, that make Lane 4 look worth seven weeks:

1. **Search volume won't do the job** — its correlation with actual impressions is ~0.001. The "obvious" prioritization signal carries no signal, so the queue must be built from behavioral evidence instead.
2. **The gap is real and content-type-shaped** — at the same page-1 position, comparison articles average ~0.14% CTR vs ~0.35% for keyword articles (2.5x difference, n=226 vs n=8,186). Position alone doesn't explain who gets clicked.
3. **Under-clickers aren't marginal pages** — the flagged pool's median CTR is ~0.17% vs ~0.73% for their visible peers at the same positions, a ~4x gap sitting on ~91M 90-day impressions.

In [3]:
# Number 1 — the obvious signal is empty
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"1) corr(search_volume, impressions_90d) = {corr:.3f}  -> not a usable priority signal")

# Number 2 — same position, very different click capture (from section 1's table)
cmp_ctr = by_type.loc["comparison article", "mean_ctr_pct"]
kw_ctr = by_type.loc["keyword article", "mean_ctr_pct"]
print(f"2) page-1 mean CTR: comparison {cmp_ctr:.2f}% vs keyword {kw_ctr:.2f}%  "
      f"({kw_ctr / cmp_ctr:.1f}x gap at the SAME position tier)")

# Number 3 — the under-clickers vs their visible peers at the same positions
peers = df[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0) & (df["avg_position"] <= 20)
    & (df["ctr"] >= 0.5)
]
print(f"3) median CTR: flagged pool {pool['ctr'].median():.2f}% vs peers {peers['ctr'].median():.2f}%"
      f"  ({len(pool):,} pages, {int(pool['impressions_90d'].sum()):,} 90d impressions at stake)")

1) corr(search_volume, impressions_90d) = 0.001  -> not a usable priority signal
2) page-1 mean CTR: comparison 0.14% vs keyword 0.35%  (2.5x gap at the SAME position tier)
3) median CTR: flagged pool 0.17% vs peers 0.73%  (9,759 pages, 90,968,008 90d impressions at stake)


## 4. Careful words: what I can and can't claim

**I can say (observed / directional / decision-support):**
- "In this pseudonymized dataset, pages of type X *observed* lower CTR than peers at the same position tier."
- "This ranking is *decision support*: it orders candidates for human review; the editor decides."
- "The model's ranking was *measured* better/worse than the transparent rule at precision@50 on held-out clients."

**I can never say:**
- **Causal claims** — "rewriting the title will raise CTR." Proving that needs an experiment; I'm ranking review candidates, not promising outcomes.
- **Google claims** — "this is a Google ranking factor." This data observes one inventory; it cannot see the algorithm.
- **Cross-position CTR comparisons without adjustment** — a 0.3% CTR is normal at position 40 and alarming at position 3; every CTR claim must be position-conditioned (the lane guide lists this as the lane's #1 mistake).
- **Anything from thin cells** — the code below shows some content-type x tier cells have n <= 10; those stay out of every claim.
- **Anything identifying** — no client names, domains, URLs, or raw queries, anywhere, ever.

**Label discipline (the trap I'm avoiding):** a page's *current* CTR gap is my **score**, not a training label — if I defined "underperformer" by my own threshold and then trained on it, the model would just learn my threshold back (the leakage lesson from notebook 02). For supervised validation, the plan (Week 3+, on the warehouse daily facts) is a **future-window observed outcome**: features from a prior window -> CTR/clicks movement in the *next* 30 days, with client-holdout splits so no client appears in both train and test.

In [4]:
# Backing the caveats with numbers
# (a) thin cells I will NOT draw conclusions from:
counts = (df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)]
          .pivot_table(values="ctr", index="position_tier", columns="content_type", aggfunc="count")
          .fillna(0).astype(int))
print("Sample size per position_tier x content_type cell:")
print(counts.to_string())
thin = int((counts <= 10).sum().sum())
print(f"-> {thin} cells have n <= 10: excluded from every claim.")
print()

# (b) rows with no position data — excluded, never treated as 'position zero':
print(f"avg_position == 0 ('no data') rows excluded from position analyses: {(df['avg_position'] == 0).sum():,}")

# (c) scale sanity: rates are x100 percentages (0.23 = 0.23%); the 100% max is a
# real edge case (a page whose every impression got a click, at tiny volume)
print(f"ctr range in the data: {df['ctr'].min():.2f}% to {df['ctr'].max():.2f}%")

Sample size per position_tier x content_type cell:
content_type   comparison article  feedly article  keyword article
position_tier                                                     
deep                            0              10              869
page_1                        226             221             8186
page_3_5                       63              41             5954
striking                       76              75             5752
top_3                           1               5              527
-> 4 cells have n <= 10: excluded from every claim.

avg_position == 0 ('no data') rows excluded from position analyses: 1,205
ctr range in the data: 0.00% to 100.00%


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.